# Module 06 — Calibration Check (Cycle 0)

**Purpose:** Evaluate the calibration quality of the Cycle 0 deep ensemble. A well-calibrated model's uncertainty estimate should match the actual prediction error.

**目的：** 评估 Cycle 0 深度集成的校准质量。校准良好的模型，其不确定性估计应与实际预测误差相符。

---

## Why Calibration Matters for Active Learning

The BALD acquisition function (Module 07) selects experiments by *mutual information* between model outputs and parameters. BALD relies on the ensemble's uncertainty being **well-calibrated** — if the model claims 0.1 std everywhere but the actual residuals are 0.5, BALD will pick experiments near the wrong decision boundary.

If ECE > 0.1, we apply **temperature scaling** (Guo et al., 2017): divide all logits by a learnable temperature T > 1 to widen the distribution and re-calibrate.

## 为什么校准对主动学习至关重要

Module 07 的 BALD acquisition function 通过**互信息**选择实验。BALD 依赖集成的不确定性**校准良好**——如果模型到处声称 std=0.1，但实际残差是 0.5，BALD 会在错误的决策边界附近选择实验。

如果 ECE > 0.1，我们应用**温度缩放**（Guo et al., 2017）：将所有 logit 除以可学习温度 T > 1，拓宽分布以重新校准。

---

**References / 参考文献:**
- Guo et al. (2017) "On Calibration of Modern Neural Networks." ICML.
- Houlsby et al. (2011) "Bayesian Active Learning for Classification and Preference Learning." arXiv:1112.5745. (BALD)

In [ ]:
# Cell 2: Imports, paths, library versions
# 第 2 单元：导入、路径、库版本

import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Path anchoring / 路径锚定
NOTEBOOK_DIR = Path.cwd().resolve()
MODULE_DIR   = NOTEBOOK_DIR
if NOTEBOOK_DIR.name == 'processes':
    MODULE_DIR = NOTEBOOK_DIR.parent
elif NOTEBOOK_DIR.name != '06_uncertainty_model':
    for p in NOTEBOOK_DIR.parents:
        if p.name == '06_uncertainty_model':
            MODULE_DIR = p
            break

PROCESSES_DIR = MODULE_DIR / 'processes'
OUTPUTS_DIR   = MODULE_DIR / 'outputs' / 'cycle_0'

if str(PROCESSES_DIR) not in sys.path:
    sys.path.insert(0, str(PROCESSES_DIR))

from ensemble import DeepEnsemble, EnsembleMember
from synthetic_data import generate_synthetic_dataset
from run_calibration import compute_regression_ece, plot_reliability_diagram

# Version printout / 版本输出
print(f'Python:     {sys.version}')
print(f'PyTorch:    {torch.__version__}')
print(f'NumPy:      {np.__version__}')
print(f'Outputs at: {OUTPUTS_DIR}')

In [ ]:
# Cell 3: Load ensemble + metadata from cycle_0 outputs
# 第 3 单元：加载 cycle_0 输出中的集成和元数据

meta_path = OUTPUTS_DIR / 'model_meta.json'
with open(meta_path) as f:
    meta = json.load(f)

print('Model metadata / 模型元数据:')
print(json.dumps({k: v for k, v in meta.items() if k not in ('train_size', 'val_size')}, indent=2))

input_dim  = meta['input_dim']
hidden_dim = meta['hidden_dim']
n_members  = meta['n_members']
embed_dim  = meta['embed_dim_per_protein']

# Reconstruct ensemble and load state_dicts
# 重建集成并加载 state_dict
ens = DeepEnsemble(input_dim=input_dim, hidden_dim=hidden_dim, n_members=n_members)
members = []
for i in range(n_members):
    pt_path = OUTPUTS_DIR / f'ensemble_member_{i}.pt'
    m = EnsembleMember(input_dim=input_dim, hidden_dim=hidden_dim)
    m.load_state_dict(torch.load(pt_path, map_location='cpu', weights_only=True))
    m.eval()
    members.append(m)
    print(f'Loaded: {pt_path.name}')
ens.members = members

In [ ]:
# Cell 4: Re-generate same synthetic dataset (same seed = same pairs + labels)
# 第 4 单元：用相同种子重新生成合成数据集（相同 pair + 标签）

SEED = 42
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    X, y, data_meta = generate_synthetic_dataset(
        seed=SEED, embed_dim=embed_dim,
    )

# Same split as training / 与训练相同的拆分
rng = np.random.default_rng(SEED)
idx = rng.permutation(len(X))
split = int(0.8 * len(X))
val_idx = idx[split:]
X_val, y_val = X[val_idx], y[val_idx]
val_meta = data_meta.iloc[val_idx].reset_index(drop=True)

print(f'Val set: {len(X_val)} pairs')
print(f'y_val stats: mean={y_val.mean():.3f}, std={y_val.std():.3f}')

In [ ]:
# Cell 5: Compute calibration metrics
# 第 5 单元：计算校准指标

mean_pred, std_pred = ens.predict(X_val)  # total_std (epistemic + aleatoric)

residuals = y_val - mean_pred
rmse = float(np.sqrt(np.mean(residuals**2)))
mae  = float(np.mean(np.abs(residuals)))

ece, bin_pred, bin_actual = compute_regression_ece(y_val, mean_pred, std_pred, n_bins=8)

print('=== Calibration Metrics / 校准指标 ===')
print(f'RMSE:            {rmse:.4f}')
print(f'MAE:             {mae:.4f}')
print(f'Mean predicted std: {std_pred.mean():.4f}')
print(f'ECE (regression):   {ece:.4f}')

if ece > 0.1:
    print(f'\n⚠ ECE = {ece:.4f} > 0.1 — consider temperature scaling.')
    print('This is expected on small (n=16) synthetic val sets.')
    print('Re-evaluate after real ELISA data arrives.')
else:
    print(f'\n✓ ECE = {ece:.4f} ≤ 0.1 — ensemble is well-calibrated.')

In [ ]:
# Cell 6: Plot reliability diagram → save calibration.png
# 第 6 单元：绘制 reliability diagram → 保存 calibration.png

plot_reliability_diagram(
    y_true=y_val,
    mean_pred=mean_pred,
    std_pred=std_pred,
    ece=ece,
    save_path=OUTPUTS_DIR / 'calibration.png',
    cycle=0,
)

# Display inline in notebook / 在 notebook 中内联显示
from IPython.display import Image, display
display(Image(filename=str(OUTPUTS_DIR / 'calibration.png')))

## Calibration Conclusion

### Cycle 0 (Synthetic Data)

The ECE reported above is computed on a **tiny validation set (n=16)** of **fully random synthetic data**. This calibration result does NOT generalize to real ELISA binding data — it is a pipeline check only.

If ECE > 0.1:
1. **Expected for synthetic data** — the ensemble has never seen biologically meaningful signal, so its sigma estimates may not align with residual magnitudes.
2. **After real ELISA arrives** — retrain on `08_cycle_data/outputs/cycle_0/elisa_processed.csv`, rerun this notebook, apply temperature scaling if ECE > 0.1.
3. **Ready for BALD (Module 07)?** — Once ECE < 0.1 on real data, the ensemble's uncertainty is reliable enough for BALD acquisition.

### Cycle 0（合成数据）

上面报告的 ECE 是在 **极小验证集（n=16）** 的 **完全随机合成数据** 上计算的。这个校准结果**不适用**于真实 ELISA 结合数据——仅作为 pipeline 检查。

如果 ECE > 0.1：
1. **合成数据的预期现象** — 集成从未见过生物学有意义的信号，因此 sigma 估计可能与残差量级不一致。
2. **真实 ELISA 到来后** — 在 `08_cycle_data/outputs/cycle_0/elisa_processed.csv` 上重训，重跑此 notebook，如 ECE > 0.1 应用温度缩放。
3. **是否可以用 BALD（Module 07）？** — 在真实数据上 ECE < 0.1 后，集成的不确定性足够可靠，可用于 BALD acquisition。